# 2-GPU DDP throughput TEST (experiment — run on the spare 4h account)

Decides whether data-parallel across both T4s actually speeds up our 4-bit QLoRA.
If yes (~2x throughput), I wire DDP into the real trainer. If it errors, **paste me
the output of the last cell** and we stay single-GPU.

Setup: **Accelerator = GPU T4 x2**, **Internet On**, Secret **HF_TOKEN**. Run cells
top to bottom. Uses DUMMY data — measures speed only, trains nothing real. ~10 min
(mostly the model download), well inside 4h.

NOTE: do NOT create any CUDA tensor in a normal cell before the launcher cell —
`notebook_launcher` forks the process and a pre-initialized CUDA context breaks it.
(The GPU-check cell uses only `device_count()`, which is safe.)

In [ ]:
# Cell 1 — deps (subprocess only; do NOT import torch/bitsandbytes here — they
# initialize CUDA on import and break notebook_launcher's fork)
import subprocess, sys
def pip(*a): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=5.8", "accelerate>=1.2", "peft>=0.14", "bitsandbytes>=0.45", "sentencepiece", "protobuf")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
print("deps installed (versions are checked inside the worker, not here)")

In [ ]:
# Cell 2 — GPU check via nvidia-smi ONLY (no torch import in the main process)
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(out)
assert out.count("
") + 1 >= 2, "Need 2 GPUs — set Accelerator to GPU T4 x2"
print("GPUs:", out.count("
") + 1)

In [ ]:
# Cell 3 — HF login + download base model (no CUDA here)
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download
login(UserSecretsClient().get_secret("HF_TOKEN"))
BASE = "/kaggle/working/e2b_test"
snapshot_download(repo_id="google/gemma-4-E2B-it-qat-q4_0-unquantized", local_dir=BASE,
                  allow_patterns=["*.json","*.safetensors","*.model","*.txt","tokenizer*"])
print("base at", BASE, "|", len(os.listdir(BASE)), "files")

In [ ]:
# Cell 4 — the timed worker (runs inside the launcher, once per process)
# Loads 4-bit E2B + RANK=16 attn-only LoRA on THIS process's GPU, times 10 steps on
# dummy data. world=1 -> single-GPU baseline; world=2 -> data-parallel across both.
def worker():
    import time, torch
    from accelerate import Accelerator
    from transformers import AutoModelForImageTextToText, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    acc = Accelerator()
    try:
        quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
        model = AutoModelForImageTextToText.from_pretrained(
            BASE, local_files_only=True, quantization_config=quant,
            torch_dtype=torch.bfloat16, device_map={"": acc.process_index})
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model = get_peft_model(model, LoraConfig(
            task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj)$"))
        model.train()
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
        model, opt = acc.prepare(model, opt)
        ids = torch.randint(0, 1000, (1, 512), device=acc.device)
        for _ in range(3):  # warmup
            loss = model(input_ids=ids, labels=ids).loss; acc.backward(loss); opt.step(); opt.zero_grad()
        torch.cuda.synchronize(); t = time.time()
        for _ in range(10):
            loss = model(input_ids=ids, labels=ids).loss; acc.backward(loss); opt.step(); opt.zero_grad()
        torch.cuda.synchronize()
        sps = (time.time() - t) / 10
        acc.print(f"  world={acc.num_processes}  {sps:.2f}s/step/proc  ->  throughput {acc.num_processes/sps:.3f} ex/s")
    except Exception as e:
        acc.print(f"  world={acc.num_processes}  ERROR: {type(e).__name__}: {e}")
print("worker defined")

In [ ]:
# Cell 5 — run both. Compare 'ex/s': if 2-GPU ex/s ~= 2x the 1-GPU ex/s, DDP gives a real speedup.
from accelerate import notebook_launcher
# 2-GPU (forked) MUST run first: a num_processes=1 launch runs in THIS process and
# initializes Accelerator state, which then blocks the 2-GPU launch.
print(">>> 2 GPU DDP");       notebook_launcher(worker, num_processes=2)
print(">>> 1 GPU baseline");  notebook_launcher(worker, num_processes=1)
print("
Read the two 'ex/s' numbers. >1.6x = worth wiring into the trainer. "
      "If 2-GPU errored, restart kernel and run ONLY cell 5's 2-GPU line, then paste the error.")